# Pensioni PA DAG — notebook v0

Notebook di validazione minima del candidate in DI.

- perimetro: snapshot regionale di dicembre 2024
- trattamenti inclusi: `DIRETTA` e `INDIRETTA/REVERSIBILE`
- scopo: controlli base su mart e prima visualizzazione, non analisi pubblica


In [1]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

SLUG = "pensioni_pa_dag_2024"
MART_TABLE = "mart_regioni_dicembre"
METRICA = "numero_partite"
DIM = "regione"

cwd = Path.cwd().resolve()
if (cwd / "dataset.yml").exists():
    candidate_dir = cwd
elif (cwd.parent / "dataset.yml").exists():
    candidate_dir = cwd.parent
else:
    raise FileNotFoundError(f"Dataset dir non trovato da cwd={cwd}")

MART_PATH = candidate_dir.parents[1] / "out" / "data" / "mart" / SLUG / "2024"
if not MART_PATH.exists():
    raise FileNotFoundError(f"Mart non trovato: {MART_PATH}. Eseguire prima toolkit run all.")

PARQUET_PATH = MART_PATH / f"{MART_TABLE}.parquet"
if not PARQUET_PATH.exists():
    raise FileNotFoundError(f"Tabella mart non trovata: {PARQUET_PATH}")

print(f"Candidate dir: {candidate_dir.name}")
print(f"Mart table: {PARQUET_PATH.name}")

Candidate dir: pensioni-pa-dag
Mart table: mart_regioni_dicembre.parquet


In [2]:
con = duckdb.connect()
df = con.execute("SELECT * FROM read_parquet(?)", [str(PARQUET_PATH)]).df()
print(f"Shape: {df.shape}")
display(df.dtypes)

Shape: (40, 7)


anno                            int32
mese                            int32
regione                           str
tipo_pensione                     str
numero_partite                float64
importi_mensili_pagati_eur    float64
righe_granulari                 int64
dtype: object

In [3]:
display(df.head(10))

,anno,mese,regione,tipo_pensione,numero_partite,importi_mensili_pagati_eur,righe_granulari
0,2024,12,Lazio,INDIRETTA/REVERSIBILE,8114.0,6561113.16,274
1,2024,12,Lazio,DIRETTA,5560.0,9167533.93,109
2,2024,12,Campania,INDIRETTA/REVERSIBILE,4476.0,4389313.18,271
3,2024,12,Sicilia,INDIRETTA/REVERSIBILE,4212.0,4021378.18,327
4,2024,12,Campania,DIRETTA,3901.0,7749487.21,89
5,2024,12,Lombardia,INDIRETTA/REVERSIBILE,3801.0,3640170.58,322
6,2024,12,Sicilia,DIRETTA,3421.0,6773875.46,104
7,2024,12,Veneto,INDIRETTA/REVERSIBILE,3252.0,2929545.66,299
8,2024,12,Emilia-Romagna,INDIRETTA/REVERSIBILE,3209.0,2526898.79,340
9,2024,12,Lombardia,DIRETTA,3002.0,5464897.39,128


In [4]:
print("Null per colonna:")
display(df.isnull().sum())

if METRICA in df.columns:
    print(f"\nRange {METRICA}: min={df[METRICA].min()}, max={df[METRICA].max()}, negativi={(df[METRICA] < 0).sum()}")
else:
    raise KeyError(f"Colonna metrica non trovata: {METRICA}")

Null per colonna:


anno                          0
mese                          0
regione                       0
tipo_pensione                 0
numero_partite                0
importi_mensili_pagati_eur    0
righe_granulari               0
dtype: int64


Range numero_partite: min=43.0, max=8114.0, negativi=0


In [5]:
(
    df.groupby(DIM)[METRICA]
    .sum()
    .sort_values(ascending=False)
    .plot(kind="bar", figsize=(12, 5), title=f"{METRICA} per {DIM}")
)
plt.tight_layout()
plt.show()

## Note v0

- Slug: `pensioni_pa_dag_2024` | Tabella mart: `mart_regioni_dicembre`
- Metrica guida: `numero_partite`
- Perimetro: snapshot regionale di dicembre 2024, solo `DIRETTA` e `INDIRETTA/REVERSIBILE`
- Questo notebook e' esplorativo e resta in DI, non e' l'analisi pubblica.